In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
from PIL import Image, UnidentifiedImageError
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

PROJECT_ROOT = "/content/drive/MyDrive/intel-image-classifier"
sys.path.insert(0, f"{PROJECT_ROOT}/src")
os.chdir(PROJECT_ROOT)
print("Connected to project folder.")

from preprocessing import get_transforms
from cnn_model import SceneCNN
from resnet_model import build_resnet18

def load_trained_model(model_type: str, checkpoint_path: str, device):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found at: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    class_names = checkpoint['class_names']
    img_size = checkpoint['img_size']

    if model_type == "cnn":
        model = SceneCNN(num_classes=len(class_names)).to(device)
    elif model_type == "resnet18":
        model = build_resnet18(num_classes=len(class_names), freeze_backbone=False).to(device)
    else:
        raise ValueError("model_type must be 'cnn' or 'resnet18'")

    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    _, transform = get_transforms(img_size=img_size)

    return model, transform, class_names

def predict_image(image_path: str, model, transform, class_names, device):
    # loads an image, runs deterministic inference without gradients, and returns predicted class + confidence dictionary for all 6 classes
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image file does not exist: {image_path}")

    try:
        image = Image.open(image_path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        raise ValueError(f"Unsupported or corrupted image file '{image_path}': {e}")

    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        probabilities = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()

    scores = {cls: float(prob) for cls, prob in zip(class_names, probabilities)}
    predicted_class = max(scores, key=scores.get)

    return predicted_class, scores

def demo_unlabeled_predictions(pred_dir: str, model_type: str = "resnet18", checkpoint_path: str = "models/resnet18_best.pth"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, transform, class_names = load_trained_model(model_type, checkpoint_path, device)

    sample_paths = sorted(glob.glob(os.path.join(pred_dir, "*.jpg")))[:10]
    if not sample_paths:
        print(f"No .jpg images found in {pred_dir}")
        return

    os.makedirs("outputs/plots", exist_ok=True)
    plt.figure(figsize=(15, 6))

    for idx, path in enumerate(sample_paths):
        pred_cls, all_scores = predict_image(path, model, transform, class_names, device)
        top_conf = all_scores[pred_cls]
        print(f"Image: {os.path.basename(path)} -> Predicted: {pred_cls} ({top_conf:.2%}) | All scores: { {k: round(v, 3) for k, v in all_scores.items()} }")

        img = Image.open(path)
        plt.subplot(2, 5, idx + 1)
        plt.imshow(img)
        plt.title(f"Pred: {pred_cls}\nConf: {top_conf:.1%}", fontsize=9)
        plt.axis("off")

    plt.suptitle(f"Unlabeled Prediction Demo ({model_type.upper()})", fontsize=12)
    plt.tight_layout()
    plt.savefig("outputs/plots/unlabeled_predictions_demo.png")
    plt.close()

if __name__ == "__main__":
    demo_unlabeled_predictions("./data/seg_pred/seg_pred", model_type="resnet18", checkpoint_path="models/resnet18_best.pth")

Image: 10004.jpg -> Predicted: street (84.08%) | All scores: {'buildings': 0.159, 'forest': 0.0, 'glacier': 0.0, 'mountain': 0.0, 'sea': 0.0, 'street': 0.841}
Image: 10005.jpg -> Predicted: mountain (79.42%) | All scores: {'buildings': 0.0, 'forest': 0.001, 'glacier': 0.202, 'mountain': 0.794, 'sea': 0.001, 'street': 0.002}
Image: 10012.jpg -> Predicted: street (91.94%) | All scores: {'buildings': 0.081, 'forest': 0.0, 'glacier': 0.0, 'mountain': 0.0, 'sea': 0.0, 'street': 0.919}
Image: 10013.jpg -> Predicted: mountain (93.10%) | All scores: {'buildings': 0.0, 'forest': 0.016, 'glacier': 0.053, 'mountain': 0.931, 'sea': 0.0, 'street': 0.0}
Image: 10017.jpg -> Predicted: mountain (43.13%) | All scores: {'buildings': 0.102, 'forest': 0.001, 'glacier': 0.286, 'mountain': 0.431, 'sea': 0.118, 'street': 0.063}
Image: 10021.jpg -> Predicted: forest (99.64%) | All scores: {'buildings': 0.0, 'forest': 0.996, 'glacier': 0.0, 'mountain': 0.003, 'sea': 0.0, 'street': 0.0}
Image: 1003.jpg -> Predi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob

PROJECT_ROOT = glob.glob('/content/drive/MyDrive/**/intel-image-classifier', recursive=True)[0]
SRC_DIR = os.path.join(PROJECT_ROOT, "src")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.chdir(PROJECT_ROOT)
print("Ready! Working in:", os.getcwd())

Mounted at /content/drive
Ready! Working in: /content/drive/MyDrive/intel-image-classifier
